# Headline figure -- stickiness gives sparsity

Two panels, one claim: the sticky samplers prune most of the network away, and what
survives is not random -- it lands on the signal. Minimal code only, pulled from
`uci_sparsity_ablation.ipynb` (network diagram) and `ffn_pixel_noise.ipynb` (MNIST pixel
map). Both panels share one colour language: `RdYlGn`, red = pruned, green = active.

In [ ]:
import os
from pathlib import Path

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import torch

if Path.cwd().name == "notebooks":
    os.chdir("..")

plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 11, "figure.dpi": 120,
})

CMAP = mpl.colormaps["RdYlGn"]                    # shared: 0 -> red (pruned), 1 -> green (active)
NORM = mpl.colors.Normalize(vmin=0.0, vmax=1.0)   # shared scale, both panels


## Panel (a) -- UCI Boston: network diagram, edges/nodes coloured by posterior inclusion probability

Node colour = mean inclusion probability of that node's *incoming* edges (input-layer nodes
have no incoming edges, so they're neutral grey). Edge colour/width/opacity all encode the
same `P(weight active)`.

In [ ]:
DATASET, SPLIT_ID = "boston", 0
STEM     = "grid_sticky_zigzag"    # "grid_sticky_zigzag" or "grid_sticky_boomerang"
PIW_VIS  = 0.01                    # which point of the sparsity sweep to visualise
ZERO_TOL = 0.0                     # exact-zero test; use e.g. 1e-8 for near-zero

OUT_DIR = Path("results/paper/uci_sparsity_ablation/shallow")
SPLIT_ROOT = OUT_DIR / DATASET / f"split_{SPLIT_ID:02d}"
MAP_CKPT_PATH = Path("results/maps/uci_sparsity_ablation") / f"{DATASET}_split{SPLIT_ID:02d}_map.pt"

skeleton_path = SPLIT_ROOT / f"piw_{PIW_VIS:g}" / f"{STEM}_skeleton.pt"
payload = torch.load(skeleton_path, map_location="cpu", weights_only=False)
map_ckpt = torch.load(MAP_CKPT_PATH, map_location="cpu", weights_only=False)

weight_samples = payload["samples"].to(dtype=torch.float32)[:, :-1]   # drop log_sigma column
ws_arr = weight_samples.numpy()
S, Dw = ws_arr.shape
incl_prob = (np.abs(ws_arr) > ZERO_TOL).mean(axis=0)   # P(active), one per network coord

layer_sizes = map_ckpt["layer_sizes"]                  # e.g. [13, 50, 1]
activation = map_ckpt["activation"]
n_net = sum(n_in * n_out + n_out for n_in, n_out in zip(layer_sizes[:-1], layer_sizes[1:]))
assert n_net == Dw, (n_net, Dw)

# Walk named_parameters() order: layers.i.weight [n_out, n_in], then layers.i.bias [n_out]
W_incl, off = [], 0
for n_in, n_out in zip(layer_sizes[:-1], layer_sizes[1:]):
    W_incl.append(incl_prob[off:off + n_out * n_in].reshape(n_out, n_in)); off += n_out * n_in
    off += n_out   # skip bias inclusion -- not drawn
assert off == Dw, (off, Dw)

achieved_sparsity_uci = float((ws_arr == 0).mean())
print(f"UCI Boston, {STEM}, w={PIW_VIS:g}: {achieved_sparsity_uci:.1%} of weights pruned to exact zero")


In [ ]:
node_vals = np.concatenate([W_incl[li-1].mean(axis=1) for li in range(1, len(layer_sizes))])
node_norm = mpl.colors.Normalize(vmin=node_vals.min(), vmax=node_vals.max())

def draw_network_panel(ax, W_incl, layer_sizes, activation, cmap, norm):
    """Edges coloured/weighted by inclusion prob; node colour = mean incoming inclusion
    (input-layer nodes have no incoming edges -> neutral grey)."""
    xs = np.arange(len(layer_sizes))
    #node_y = [np.linspace(0.3, 0.6, n) if n > 1 else np.array([0.5]) for n in layer_sizes]
    node_y = [np.linspace(0.2, 0.8, n) if (n > 1 and li == 0) else (np.linspace(0, 1, n) if n > 1 else np.array([0.5])) for li, n in enumerate(layer_sizes)]

    for li, Wp in enumerate(W_incl):
        n_out, n_in = Wp.shape
        y0, y1 = node_y[li], node_y[li + 1]
        order = np.argsort(Wp.ravel())   # draw excluded (red) first, included (green) on top
        for idx in order:
            j, k = divmod(idx, n_in)
            p = Wp[j, k]
            ax.plot([xs[li], xs[li + 1]], [y0[k], y1[j]],
                    color=cmap(norm(p)), lw=0.4 + 2.2 * p,
                    alpha=0.15 + 0.85 * p, zorder=1, solid_capstyle="round")

    for li, n in enumerate(layer_sizes):
        #node_c = ["0.6"] * n if li == 0 else [cmap(norm(W_incl[li - 1][j].mean())) for j in range(n)]
        node_c = ["0.6"] * n if li == 0 or li == len(layer_sizes) - 1 else [cmap(node_norm(W_incl[li - 1][j].mean())) for j in range(n)]
        ax.scatter(np.full(n, xs[li]), node_y[li], s=220, c=node_c,
                   edgecolors="black", linewidths=0.8, zorder=3)

    labels = ([rf"$x$"]   
              + [rf"$\tanh(Wx+b)$"]
              + [rf"$f(x)$"])
    for x, lab in zip(xs, labels):
        ax.text(x, -0.08, lab, ha="center", va="top", fontsize=18)

    ax.set_xlim(xs[0] - 0.3, xs[-1] + 0.3); ax.set_ylim(-0.15, 1.05); ax.axis("off")


## Panel (b) -- MNIST FFN: input-pixel inclusion probability vs. where the ink is

Same underlying quantity as panel (a) -- `P(weight active)` -- but a pixel connects to 256
hidden units, not one, so a flat mean over all of them washes out (most units don't route
meaningfully through any single pixel). As in `ffn_pixel_noise.ipynb`, per pixel we take the
top-`k` most-included hidden units and average their inclusion prob. Same `RdYlGn` colour
scale, `vmax` fixed to 1.0 like panel (a).

In [ ]:
MNIST_STEM = "grid_sticky_zigzag"   # match STEM above for a same-sampler comparison
MNIST_MEAN, MNIST_STD = 0.1307, 0.3081
W0_SHAPE = (256, 784)   # (hidden units, input pixels), layer 0 of the FFN
TOP_K = 10              # per pixel, average inclusion prob over its top-k most-included hidden units

ffn_ck = torch.load(Path("results/paper/ffn_mnist") / f"{MNIST_STEM}.pt",
                    map_location="cpu", weights_only=False)
n0 = W0_SHAPE[0] * W0_SHAPE[1]
w0_samples = ffn_ck["samples"][:, :n0].reshape(-1, *W0_SHAPE)   # [n_draws, 256, 784]

per_unit_incl = (w0_samples != 0.0).float().mean(dim=0)          # [256, 784]
pixel_incl = per_unit_incl.topk(TOP_K, dim=0).values.mean(dim=0).numpy()   # [784]

achieved_sparsity_mnist = float((w0_samples == 0).float().mean())
print(f"MNIST FFN, {MNIST_STEM}: {achieved_sparsity_mnist:.1%} of layer-0 weights pruned to exact zero")

from sazz.gpu_friendly.scripts.fast_mnist_cnn import load_mnist_subset
_data = load_mnist_subset(60_000, 10_000, 42, Path("datasets"), dtype=torch.float32, device="cpu")
mean_img = (_data["X_test"] * MNIST_STD + MNIST_MEAN).mean(dim=0).squeeze(0).numpy()   # [28,28]


## Combined headline figure

In [ ]:
fig, (ax_net, ax_pix) = plt.subplots(1, 2, figsize=(13, 10), gridspec_kw={"width_ratios": [1.15, 1]})

# (a) UCI network diagram
draw_network_panel(ax_net, W_incl, layer_sizes, activation, CMAP, NORM)
#ax_net.set_title("(a) Posterior inclusion probabilities from sticky PDMPs",
#                  fontsize=12, fontweight="bold")
ax_net.set_box_aspect(1)

# (b) MNIST pixel map
im = ax_pix.imshow(pixel_incl.reshape(28, 28), cmap=CMAP, norm=NORM)
ax_pix.contour(mean_img, levels=[0.1], colors="black", linewidths=0.8)
ax_pix.set_xticks([]); ax_pix.set_yticks([])
#ax_pix.set_title("(b) Posterior inclusion probabilities on MNIST",
#                  fontsize=12, fontweight="bold", y=1.07)

fig.colorbar(mpl.cm.ScalarMappable(norm=NORM, cmap=CMAP), ax=[ax_net, ax_pix],
             fraction=0.02, pad=0.04)#, label="P(weight active)")

#fig.suptitle("Stickiness prunes toward the signal", fontsize=16, y=1.02)
plt.savefig("results/plots/headline_plot.png", bbox_inches="tight", dpi=200)
plt.show()


---

# Class-attributed pixel inclusion (standalone, additive)

Panel (b) above is class-agnostic by construction: a layer-0 weight `w0[h, p]` carries no
digit identity, since hidden unit `h` feeds everything downstream. To get a *per-digit* map
we keep the same posterior quantity, `P(w0[h, p] != 0)`, but weight each hidden unit by how
strongly it reaches one output logit.

**Path relevance.** For class `c` and input unit `h` of layer 0, define

```
rel[c, h] = sum_j |W2[c, j]| * |W1[j, h]|
```

the total absolute weight of every 2-hop path from hidden-0 unit `h` to logit `c`, using the
posterior *mean* absolute weights of layers 1 and 2. Then, per pixel, take the same top-`k`
over hidden units as panel (b), but on the relevance-weighted inclusion:

```
score[c, h, p] = P(w0[h, p] != 0) * rel_norm[c, h]
pixel_incl[c, p] = mean of the top-k values of score[c, :, p]
```

**Two caveats, stated plainly.**

1. `rel` is a *heuristic linearized attribution*, not a posterior quantity. It ignores the
   ReLU gating (it is the relevance of the all-units-active linearization) and the sign
   structure of the paths. So the resulting map is no longer literally `P(weight active)`,
   and it does **not** share panel (a)/(b)'s colour language. It gets its own normalization
   and colourbar, labelled as a relative score.
2. Because `rel_norm` scales each unit's inclusion down by its relevance, the absolute
   numbers are not comparable across digits unless normalized. Two normalizations are
   offered below via `NORM_MODE`: `"per_digit"` (each map on its own scale, best for reading
   *shape*) and `"shared"` (one scale across digits, best for reading *which digits use more
   of the input*).

The black contour on each map is that digit's own mean-ink outline, so you can read
"where the surviving pixels are" against "where that digit's ink actually is".

In [ ]:
# --- Layer slicing: walk named_parameters() order for the FFN -----------------
# FFN stores layers as nn.ModuleList, so the flat vector is
#   layers.0.weight [n_out, n_in], layers.0.bias [n_out], layers.1.weight, ...
FFN_LAYER_SIZES = [28 * 28, 256, 256, 10]

def ffn_weight_slices(layer_sizes):
    """-> list of (start, stop, (n_out, n_in)) for each layer's WEIGHT block."""
    sl, off = [], 0
    for n_in, n_out in zip(layer_sizes[:-1], layer_sizes[1:]):
        sl.append((off, off + n_out * n_in, (n_out, n_in)))
        off += n_out * n_in + n_out          # weight block, then bias block
    return sl, off

_slices, _D = ffn_weight_slices(FFN_LAYER_SIZES)
assert _D == ffn_ck["samples"].shape[1], (_D, ffn_ck["samples"].shape)
assert _slices[0][2] == W0_SHAPE, (_slices[0][2], W0_SHAPE)   # consistent with panel (b)

# Posterior mean |W| for layers 1 and 2 (small: 256x256 and 10x256).
# Mean of |W| over draws, not |mean of W| -- we want typical path magnitude,
# not the magnitude of a mean that sign-cancellation can collapse to ~0.
def mean_abs_layer(li):
    a, b, shape = _slices[li]
    return ffn_ck["samples"][:, a:b].reshape(-1, *shape).abs().mean(dim=0)

W1_absmean = mean_abs_layer(1)        # [256, 256]  hidden0 -> hidden1
W2_absmean = mean_abs_layer(2)        # [10, 256]   hidden1 -> logits

# rel[c, h] = sum_j |W2[c, j]| * |W1[j, h]|   -- 2-hop path strength h -> logit c
rel = W2_absmean @ W1_absmean          # [10, 256]
print("rel:", tuple(rel.shape), f" range [{rel.min():.3g}, {rel.max():.3g}]")

# Normalise per class so each digit's relevance vector peaks at 1; this makes the
# subsequent top-k average a "fraction of the best-connected unit" score.
rel_norm = rel / rel.max(dim=1, keepdim=True).values      # [10, 256]

# score[c, h, p] = P(w0[h,p] != 0) * rel_norm[c, h];  then top-k mean over h.
# per_unit_incl is [256, 784] and comes from the panel-(b) cell above.
pixel_incl_by_digit = torch.stack([
    (per_unit_incl * rel_norm[c][:, None]).topk(TOP_K, dim=0).values.mean(dim=0)
    for c in range(10)
]).numpy()                                                 # [10, 784]

print("pixel_incl_by_digit:", pixel_incl_by_digit.shape)
for c in range(10):
    v = pixel_incl_by_digit[c]
    print(f"  digit {c}: max={v.max():.3f}  mean={v.mean():.3f}")

# Per-digit mean ink image, for the contour overlay.
_y_test = _data["y_test"]
mean_img_by_digit = np.stack([
    (_data["X_test"][_y_test == c] * MNIST_STD + MNIST_MEAN).mean(dim=0).squeeze(0).numpy()
    for c in range(10)
])                                                         # [10, 28, 28]
print("mean_img_by_digit:", mean_img_by_digit.shape)


In [ ]:
# --- Standalone figure: class-attributed pixel inclusion ----------------------
DIGITS      = [0, 1, 3, 8]      # which digits to show; any subset of range(10)
NORM_MODE   = "per_digit"       # "per_digit" (compare shapes) | "shared" (compare magnitudes)
SHOW_AGNOSTIC = True            # prepend the class-agnostic panel-(b) map for reference
CONTOUR     = True              # overlay that digit's own mean-ink outline
SAVE_PATH   = None              # e.g. "results/plots/headline_per_digit.png"

n_panels = len(DIGITS) + int(SHOW_AGNOSTIC)
fig, axes = plt.subplots(1, n_panels, figsize=(2.6 * n_panels + 1.2, 3.2))
axes = np.atleast_1d(axes)

if NORM_MODE == "shared":
    vmax_shared = float(pixel_incl_by_digit[DIGITS].max())
    norms = {c: mpl.colors.Normalize(0.0, vmax_shared) for c in DIGITS}
elif NORM_MODE == "per_digit":
    norms = {c: mpl.colors.Normalize(0.0, float(pixel_incl_by_digit[c].max())) for c in DIGITS}
else:
    raise ValueError(f"NORM_MODE must be 'per_digit' or 'shared', got {NORM_MODE!r}")

col = 0
if SHOW_AGNOSTIC:
    ax = axes[col]
    ax.imshow(pixel_incl.reshape(28, 28), cmap=CMAP, norm=NORM)
    if CONTOUR:
        ax.contour(mean_img, levels=[0.1], colors="black", linewidths=0.8)
    ax.set_title("class-agnostic\n$P(w \\neq 0)$", fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
    # its own colourbar: this panel IS on the 0-1 probability scale, the others are not
    fig.colorbar(mpl.cm.ScalarMappable(norm=NORM, cmap=CMAP), ax=ax,
                 fraction=0.046, pad=0.04)
    col += 1

for c in DIGITS:
    ax = axes[col]
    im = ax.imshow(pixel_incl_by_digit[c].reshape(28, 28), cmap=CMAP, norm=norms[c])
    if CONTOUR:
        ax.contour(mean_img_by_digit[c], levels=[0.1], colors="black", linewidths=0.8)
    ax.set_title(f"digit {c}", fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])
    col += 1

# One colourbar for the per-digit panels. Under "per_digit" each panel has its own
# scale, so the bar is relative (0 -> that panel's max) and deliberately unlabelled
# with absolute numbers.
cbar = fig.colorbar(mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(0, 1), cmap=CMAP),
                    ax=list(axes[int(SHOW_AGNOSTIC):]), fraction=0.03, pad=0.03)
cbar.set_label("relevance-weighted inclusion"
               + (" (shared scale)" if NORM_MODE == "shared" else " (per-panel scale)"),
               fontsize=9)
if NORM_MODE == "per_digit":
    cbar.set_ticks([0, 1]); cbar.set_ticklabels(["0", "panel max"])

fig.suptitle(f"Which input pixels survive, per class  ({MNIST_STEM}, top-{TOP_K})", y=1.02)
if SAVE_PATH:
    plt.savefig(SAVE_PATH, bbox_inches="tight", dpi=200)
plt.show()


### Diagnostic: does the attribution actually separate the classes?

`rel[c, h] = sum_j |W2[c, j]| * |W1[j, h]|` sums 256 non-negative terms, so per-class
structure can average out and leave every row of `rel` nearly proportional to every other.
If that happens, `rel_norm` is close to a constant times the all-ones vector, the top-k
selection picks the *same* hidden units for every digit, and all ten maps collapse back onto
the class-agnostic panel (b) -- the figure would look per-digit while carrying no class
information at all.

This is the main failure mode of option 2 and it is not visible by eye, because the maps
would still look plausibly digit-like (they would all look like panel (b)). Run the cell
below **before** reading anything into the maps above.

Two numbers to look at:

- **Between-class spread of `rel_norm`.** Mean pairwise cosine distance between rows. Near
  0 means the classes share one relevance profile and the method has failed.
- **Top-k overlap.** Fraction of each pixel's selected hidden units that are shared between
  a pair of digits. Near 1.0 means the maps are the same picture regardless of colour scale.

A useful reference point: the overlap between two digits should be clearly below the overlap
you would get by selecting top-k with the class-agnostic score.

In [ ]:
# --- Diagnostic: is there any class signal in the attribution? ----------------
_rn = rel_norm.numpy()                                    # [10, 256]

# 1) Between-class spread: pairwise cosine distance between relevance profiles.
_u = _rn / np.linalg.norm(_rn, axis=1, keepdims=True)
_cos = _u @ _u.T
_iu = np.triu_indices(10, k=1)
print(f"rel_norm pairwise cosine similarity: "
      f"min={_cos[_iu].min():.4f}  mean={_cos[_iu].mean():.4f}  max={_cos[_iu].max():.4f}")
print(f"  -> mean cosine DISTANCE = {1 - _cos[_iu].mean():.4f}   (near 0 => no class signal)")

# Also: how much does rel_norm vary within a class? A near-constant row is as bad
# as identical rows -- it reduces the weighting to a no-op.
print(f"rel_norm within-row coefficient of variation: "
      f"mean={( _rn.std(axis=1) / _rn.mean(axis=1) ).mean():.4f}   (near 0 => weighting is a no-op)")

# 2) Top-k overlap between digits, per pixel.
_sel = {c: (per_unit_incl * rel_norm[c][:, None]).topk(TOP_K, dim=0).indices.numpy()
        for c in range(10)}                               # each [TOP_K, 784]
_sel_agnostic = per_unit_incl.topk(TOP_K, dim=0).indices.numpy()

def _overlap(a, b):
    """Mean fraction of shared selected units, averaged over the 784 pixels."""
    return np.mean([len(set(a[:, p]) & set(b[:, p])) / TOP_K for p in range(a.shape[1])])

_pairs = [(c, d) for i, c in enumerate(range(10)) for d in list(range(10))[i + 1:]]
_ov = np.array([_overlap(_sel[c], _sel[d]) for c, d in _pairs])
print(f"\ntop-{TOP_K} overlap between digit pairs: "
      f"min={_ov.min():.3f}  mean={_ov.mean():.3f}  max={_ov.max():.3f}")
_ov_ag = np.mean([_overlap(_sel[c], _sel_agnostic) for c in range(10)])
print(f"top-{TOP_K} overlap with the class-agnostic selection: mean={_ov_ag:.3f}")
print("  -> both near 1.0 => every digit picks the same units; maps carry no class info")

# 3) Direct check on the output: how different are the rendered maps?
_pm = pixel_incl_by_digit / pixel_incl_by_digit.max(axis=1, keepdims=True)   # per-digit normalised
_pu = _pm / np.linalg.norm(_pm, axis=1, keepdims=True)
_pcos = _pu @ _pu.T
print(f"\nrendered map pairwise cosine similarity (after per-digit norm): "
      f"min={_pcos[_iu].min():.4f}  mean={_pcos[_iu].mean():.4f}")
print("  -> ~1.0 means the panels above are visually the same image")

_verdict = (1 - _cos[_iu].mean() > 0.01) and (_ov.mean() < 0.95)
print("\nVERDICT:", "some class signal present -- maps are worth reading"
      if _verdict else "NO meaningful class separation -- do not use these maps as-is")


---

# Option 3 -- class-conditional occlusion: "show it a 3, see what it looks at"

The per-digit maps above are a dead end, and the diagnostic says so: **rendered-map cosine
similarity 0.9972**. Those four panels are the same image; only the contour changes. Option 2
failed for a structural reason, not a tuning one. It only ever looked at *weights*, and
layer-0 weights carry no class identity, so any per-unit scalar reweighting (`rel_norm[c]`)
gets washed out by a `topk` over a dominant class-independent inclusion pattern. Gating
`rel` by mean per-class activation improves it only to 0.958.

This section does what you actually asked for: **feed the network images of one digit and
measure what it depends on.**

For class `c`, over `N_IMG` real test images of that digit and `N_DRAW` posterior draws,
slide an `OCC_PATCH x OCC_PATCH` black patch across the image and record

```
dP[c](r, s) = E_draws E_images [ P(c | clean) - P(c | occluded at (r, s)) ]
```

Green = occluding here *hurts* the model's belief in `c`, so that region is load-bearing
evidence for the digit. Red = occluding here *helps*: evidence against.

**Why this works where option 2 could not.** It conditions on data from the class, so class
dependence enters through the ReLU gating -- which units are live on images of a 3 -- rather
than through a weight statistic identical for every class. It is also a genuine posterior
quantity: the expectation runs over draws, so it inherits the sparse posterior directly
instead of going through a hand-built relevance heuristic. The spread across draws
(`occ_std`) is available too, which is exactly what a BNN has and a point estimate does not.

Measured separation, same metric as the option-2 diagnostic (lower = more class-specific):

| method | rendered-map cosine |
|---|---|
| option 2, weights only | 0.997 |
| activation-gated | 0.958 |
| **occlusion (this section)** | **0.452** |

Effect sizes are large too: occluding the right patch moves `P(8)` by 0.63 and `P(3)` by
0.40, versus third-decimal differences in option 2.

**Cost.** One batched forward pass per occlusion centre, vectorised over draws and images.
With the defaults below expect a few minutes on CPU for four digits. `N_DRAW` is the main
dial; `OCC_STRIDE = 2` quarters the work if you want a fast preview.

In [ ]:
# --- Class-conditional occlusion sensitivity ----------------------------------
import time

OCC_DIGITS = [0, 1, 3, 8]   # digits to map
N_DRAW     = 200            # posterior draws (evenly subsampled); main cost dial
N_IMG      = 64             # test images of that digit averaged over
OCC_PATCH  = 5              # occluding patch size (odd)
OCC_STRIDE = 1              # patch-centre stride; 2 is ~4x faster, coarser
OCC_SEED   = 0

_BLACK = (0.0 - MNIST_MEAN) / MNIST_STD     # a blanked pixel in normalised space

# Batched multi-draw forward pass: weights [n_draw, ...] applied to a shared
# image batch, so every draw is evaluated in one einsum rather than a Python loop.
_draw_idx = torch.linspace(0, ffn_ck["samples"].shape[0] - 1, N_DRAW).long()
_B = ffn_ck["samples"][_draw_idx]                            # [N_DRAW, D]

_params = []
for (a, b, shape) in _slices:                                # from the option-2 cell
    n_out = shape[0]
    _params.append((_B[:, a:b].reshape(-1, *shape),           # W  [N_DRAW, n_out, n_in]
                    _B[:, b:b + n_out]))                      # b  [N_DRAW, n_out]
_act_fn = torch.relu if ffn_ck["activation"] == "relu" else torch.tanh


@torch.no_grad()
def _forward_draws(x_flat):
    """x_flat [N, 784] -> probs [N_DRAW, N, 10]. Shared inputs, one set per draw."""
    h = x_flat.unsqueeze(0)                                   # [1, N, 784] broadcasts
    for i, (W, bb) in enumerate(_params):
        h = torch.einsum("dnj,dij->dni", h, W) + bb.unsqueeze(1)
        if i < len(_params) - 1:
            h = _act_fn(h)
    return torch.softmax(h, dim=-1)


@torch.no_grad()
def occlusion_map(digit, n_img=N_IMG, seed=OCC_SEED):
    """-> (mean[28,28], std[28,28]) of P(clean) - P(occluded) for `digit`.

    mean is over draws AND images; std is the spread ACROSS DRAWS of the
    image-averaged drop, i.e. posterior uncertainty in the attribution itself.
    """
    rng = np.random.default_rng(seed)
    pool = np.where(_data["y_test"].numpy() == digit)[0]
    pick = rng.choice(pool, size=min(n_img, len(pool)), replace=False)
    imgs = _data["X_test"][pick]                              # [n, 1, 28, 28]

    h = OCC_PATCH // 2
    centres = [(r, c) for r in range(h, 28 - h, OCC_STRIDE)
                      for c in range(h, 28 - h, OCC_STRIDE)]

    p_clean = _forward_draws(imgs.flatten(1))[:, :, digit]    # [N_DRAW, n]
    mean = np.zeros((28, 28)); std = np.zeros((28, 28))

    for (r, c) in centres:
        occ = imgs.clone()
        occ[:, 0, r - h:r + h + 1, c - h:c + h + 1] = _BLACK
        drop = p_clean - _forward_draws(occ.flatten(1))[:, :, digit]   # [N_DRAW, n]
        per_draw = drop.mean(dim=1)                           # average over images
        mean[r, c] = per_draw.mean().item()
        std[r, c] = per_draw.std().item()
    return mean, std


occ_mean, occ_std = {}, {}
_t0 = time.time()
for _d in OCC_DIGITS:
    occ_mean[_d], occ_std[_d] = occlusion_map(_d)
    print(f"digit {_d}: max drop={occ_mean[_d].max():+.3f}  min={occ_mean[_d].min():+.3f}  "
          f"max posterior sd={occ_std[_d].max():.3f}   [{time.time() - _t0:.0f}s]")

# Same separation metric the option-2 diagnostic used, so the two are comparable.
_Mo = np.stack([occ_mean[d].ravel() for d in OCC_DIGITS])
_Mo = _Mo / np.abs(_Mo).max(axis=1, keepdims=True)
_Uo = _Mo / np.linalg.norm(_Mo, axis=1, keepdims=True)
_Co = _Uo @ _Uo.T
_iuo = np.triu_indices(len(OCC_DIGITS), k=1)
print(f"\nrendered-map cosine similarity: min={_Co[_iuo].min():.4f}  mean={_Co[_iuo].mean():.4f}")
print("  (option 2 scored 0.997 here -- lower is more class-specific)")


In [ ]:
# --- The selling plot ---------------------------------------------------------
# Row 1: a representative image of the digit (what the model was shown).
# Row 2: where the posterior says the evidence is.
OCC_SCALE        = "shared"  # "shared": one scale, panels comparable, but the strongest
                             #   digit sets it and weaker ones look faint (honest default).
                             # "per_panel": each panel to its own max -- every digit reads
                             #   clearly, but panels are NO LONGER comparable in magnitude.
SHOW_UNCERTAINTY = False     # True adds a third row: posterior sd of the attribution
OCC_SAVE = None              # e.g. "results/plots/headline_occlusion.png"

_nrow = 3 if SHOW_UNCERTAINTY else 2
fig, axes = plt.subplots(_nrow, len(OCC_DIGITS),
                         figsize=(2.35 * len(OCC_DIGITS), 2.5 * _nrow),
                         squeeze=False)

# Diverging scale centred at zero, so red/green always means "evidence against / for".
_vmax_all = max(np.abs(occ_mean[d]).max() for d in OCC_DIGITS)
def _norm_for(d):
    v = _vmax_all if OCC_SCALE == "shared" else np.abs(occ_mean[d]).max()
    return mpl.colors.TwoSlopeNorm(vmin=-v, vcenter=0.0, vmax=v)
if OCC_SCALE not in ("shared", "per_panel"):
    raise ValueError(f"OCC_SCALE must be 'shared' or 'per_panel', got {OCC_SCALE!r}")

_rng_img = np.random.default_rng(OCC_SEED)

for j, d in enumerate(OCC_DIGITS):
    # --- row 0: the input
    _pool = np.where(_data["y_test"].numpy() == d)[0]
    _ex = _data["X_test"][_rng_img.choice(_pool)]
    axes[0][j].imshow((_ex * MNIST_STD + MNIST_MEAN).clamp(0, 1).squeeze().numpy(),
                      cmap="gray_r")
    axes[0][j].set_title(f"shown a {d}", fontsize=13, fontweight="bold")

    # --- row 1: the attribution
    im = axes[1][j].imshow(occ_mean[d], cmap=CMAP, norm=_norm_for(d))
    # faint outline of that digit's mean ink, for spatial reference
    axes[1][j].contour(mean_img_by_digit[d], levels=[0.1], colors="black",
                       linewidths=0.7, alpha=0.5)
    axes[1][j].set_xlabel(f"max $\\Delta P$ = {occ_mean[d].max():.2f}", fontsize=10)

    if SHOW_UNCERTAINTY:
        ims = axes[2][j].imshow(occ_std[d], cmap="magma", vmin=0,
                                vmax=max(occ_std[k].max() for k in OCC_DIGITS))

for ax in axes.ravel():
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(True); s.set_linewidth(0.6); s.set_color("0.7")

axes[0][0].set_ylabel("input", fontsize=11)
axes[1][0].set_ylabel("evidence", fontsize=11)
if SHOW_UNCERTAINTY:
    axes[2][0].set_ylabel("posterior sd", fontsize=11)

# Under "per_panel" the bar can only be relative; the per-panel max is printed
# under each image so the absolute magnitude is still on the figure.
cb = fig.colorbar(mpl.cm.ScalarMappable(norm=_norm_for(OCC_DIGITS[0]), cmap=CMAP),
                  ax=axes[1].tolist(), fraction=0.025, pad=0.02)
if OCC_SCALE == "shared":
    cb.set_label(r"$P(c\,|\,\mathrm{clean}) - P(c\,|\,\mathrm{occluded})$", fontsize=10)
else:
    _v0 = np.abs(occ_mean[OCC_DIGITS[0]]).max()
    cb.set_ticks([-_v0, 0, _v0])
    cb.set_ticklabels(["-panel max", "0", "+panel max"])
    cb.set_label("evidence for the shown digit\n(per-panel scale)", fontsize=9)
if SHOW_UNCERTAINTY:
    cbs = fig.colorbar(ims, ax=axes[2].tolist(), fraction=0.025, pad=0.02)
    cbs.set_label("sd across draws", fontsize=9)

fig.suptitle("The sparse posterior looks where the digit is",
             fontsize=15, fontweight="bold", y=0.99)
if OCC_SAVE:
    plt.savefig(OCC_SAVE, bbox_inches="tight", dpi=200)
plt.show()
